# APK 6725 - AI for Sport & Movement Sciences
## Assignment 7: Logistic Regression on Two Datasets

This assignment practices the ideas from lecture 04.01. You will use the **same technique**, logistic regression, on **two different datasets**:

- **Part A (2.5 points):** classify a walking recording as control or Parkinson's disease (PD). Two classes.
- **Part B (2.5 points):** classify the speed of a running stride: 2.5, 3.5 or 4.5 m/s. Three classes.

The two parts have the same four steps: prepare the data, one input, two inputs, more inputs. Look for what stays the same and what changes. Write your code in the cells provided and test it.

Total: 5 points.

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
os.chdir('/content')                      # start from a valid directory
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/AIforMovementScience'
if os.path.isdir(REPO + '/.git'):
    # already downloaded before: just update it
    !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
else:
    # first time, or a broken/partial copy: download a fresh copy
    !rm -rf "$REPO"
    !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"

WORKDIR = REPO + '/Assignments'          # this notebook's folder in the course files
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)           # so local .py files (e.g. FunctionExamples.py) import correctly
DATA = REPO + '/lectures/data'        # the folder that holds this assignment's data files
print('Setup complete. You can now run the rest of the notebook.')

## Accuracy

In this assignment you measure a classifier with its **accuracy**: the fraction of samples that the model classifies correctly. For a fitted model, `model.score(X, y)` gives the accuracy. Next week you will learn better measures.

## How to submit

Before you submit, please make sure your notebook is organized and reproducible. Part of your grade is for this.

- Answer each question in its own section, using the headings already provided.
- Put your code in the code cells; when a question asks for a written answer, use the markdown answer cell.
- Delete any scratch or experimental cells that are not part of your answer.
- Download your notebook: **File > Download > Download .ipynb**, name it `YourName_Assignment_7.ipynb`, and upload it to Canvas.

---
# Part A - Parkinson's disease vs control (2.5 points)

## The data

The file **`gait_measures_PD_controls_ALL.csv`** comes from the Gait in Parkinson's Disease database (PhysioNet, <https://physionet.org/content/gaitpdb/>). You used it in Assignment 6. It is in `lectures/data`. The setup cell stored that path in the variable `DATA`.

Each row is one walking recording. The columns are:

| Column | Contents |
| --- | --- |
| `ID` | the subject who produced the recording |
| `class` | `0` for a control subject, `1` for a subject with Parkinson's disease |
| `avg_speed` | walking speed, in m/s |
| `avg_step_length` | step length, in m |
| `cadence` | steps per minute |
| `avg_stride_time_...`, `avg_stance_time_...`, `avg_swing_time_...`, `avg_step_time` | timing measures, in s |
| `cv_...` (7 columns) | variability of each timing measure (coefficient of variation, a fraction: 0.20 = 20%) |

**Goal:** predict `class` (control or PD) from the gait measures. In Assignment 6 the output was a number (speed). Now the output is a class, so you use `LogisticRegression` instead of `LinearRegression`.

## Question A1 (0.25 points) - Prepare the data

1. Read `gait_measures_PD_controls_ALL.csv` into a DataFrame called `data`. Remove every row with a missing value.
2. Print the number of recordings in each class (`data['class'].value_counts()`).
3. A very simple "model" says PD for every recording. Compute its accuracy: the fraction of recordings that are PD.
4. Make a list called `features` with every column except `ID`, `class` and `avg_speed`. Do this **before** step 5.
5. Add two columns with the same information in units that are easier to read:
   ```Python
   data['step_length_cm']   = data['avg_step_length'] * 100   # m  -> cm
   data['cv_step_time_pct'] = data['cv_step_time'] * 100      # fraction -> %
   ```
6. Split the data. `stratify` keeps the same fraction of PD recordings in both sets:
   ```Python
   from sklearn.model_selection import train_test_split
   train, test = train_test_split(data, test_size=0.25, random_state=42, stratify=data['class'])
   y_train, y_test = train['class'], test['class']
   ```

In the answer cell: what is the accuracy of the always-PD model? Why must every model in Part A do better than this number?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. read and remove rows with missing values
data = pd.read_csv(f"{DATA}/gait_measures_PD_controls_ALL.csv")
data = data.dropna()

# 2. recordings in each class
print(data['class'].value_counts())

# 3. accuracy of the always-PD model = fraction of recordings that are PD
always_pd_accuracy = (data['class'] == 1).mean()
print(f'\naccuracy of the always-PD model: {always_pd_accuracy:.3f}')

# 4. features: every column except ID, class and avg_speed (before adding the new columns)
features = [col for col in data.columns if col not in ['ID', 'class', 'avg_speed']]
print('number of features:', len(features))

# 5. same information in easier units
data['step_length_cm']   = data['avg_step_length'] * 100   # m  -> cm
data['cv_step_time_pct'] = data['cv_step_time'] * 100      # fraction -> %

# 6. stratified split
train, test = train_test_split(data, test_size=0.25, random_state=42, stratify=data['class'])
y_train, y_test = train['class'], test['class']
print(f'train rows: {len(train)} ({y_train.mean():.1%} PD), test rows: {len(test)} ({y_test.mean():.1%} PD)')

**Your answer:** There are 213 PD recordings and 92 control recordings, so the always-PD model is right for every PD recording and wrong for every control. Its accuracy is 213 / 305 = **0.698 (about 70%)**.

Every model must beat this number because this "model" learns nothing from the gait measures. It only uses the fact that the classes are unbalanced. A model that reaches 70% accuracy could be just as useless, labelling almost everyone PD. Only accuracy above 70% shows that the gait measures carry real information about PD.

## Question A2 (1 point) - One input: step length

People with PD often walk with short steps. Answer parts (a) to (c), each in its own cell.

### Part (a) - Fit the model and compute a probability by hand

Train a `LogisticRegression` on the train set, with `step_length_cm` as the only input. Print the coefficient ($\beta_1$) and the intercept ($\beta_0$).

For a subject with a step length of **55 cm** and a subject with **70 cm**:

1. Compute the probability of PD **by hand**, as in lecture 04.01: $z = \beta_0 + \beta_1 x$, then $p = 1/(1+e^{-z})$.
2. Check your result with `model.predict_proba`. Column 0 is the probability of class 0; column 1 is the probability of class 1.

**Hint:** use double brackets, `train[['step_length_cm']]`, to keep a table with one column. For `predict_proba`, use a DataFrame with the same column name: `pd.DataFrame({'step_length_cm': [55, 70]})`.

In the answer cell: is the coefficient positive or negative? What does its sign mean for walking?

In [ ]:
model_A1 = LogisticRegression()
model_A1.fit(train[['step_length_cm']], y_train)

b1 = model_A1.coef_[0, 0]
b0 = model_A1.intercept_[0]
print(f'coefficient (beta_1): {b1:.4f} per cm')
print(f'intercept   (beta_0): {b0:.4f}')

# probability of PD by hand: z = b0 + b1*x, p = 1 / (1 + e^-z)
for x in [55, 70]:
    z = b0 + b1 * x
    p = 1 / (1 + np.exp(-z))
    print(f'step length {x} cm: z = {z:.3f}, p(PD) by hand = {p:.4f}')

# check with predict_proba (column 1 = probability of class 1, PD)
check = model_A1.predict_proba(pd.DataFrame({'step_length_cm': [55, 70]}))
print('\npredict_proba [p(control), p(PD)]:')
print(check)

**Your answer:** The model gives $\beta_1$ = −0.0976 per cm and $\beta_0$ = 6.877. By hand:

- 55 cm: $z$ = 1.512, so $p$(PD) = 0.819.
- 70 cm: $z$ = 0.048, so $p$(PD) = 0.512.

`predict_proba` gives exactly the same values.

The coefficient is **negative**. The longer the step, the lower $z$, and so the lower the probability of PD. For walking, this means shorter steps go with a higher chance that the recording comes from a person with PD, which matches the short, shuffling steps typical of Parkinsonian gait. Going from 55 to 70 cm drops the probability of PD from 82% to 51%.

### Part (b) - The probability curve

1. Compute the step length where the probability of PD is 0.5. At this point $z = 0$, so $x = -\beta_0 / \beta_1$.
2. Plot the probability of PD for step lengths from 30 cm to 100 cm (use `np.linspace` and `predict_proba`). On the same axes, plot the **train** recordings as points: step length on the x-axis, `class` (0 or 1) on the y-axis. Add a vertical line at the 50% point. Label the axes.
3. Print the accuracy on the train set and on the test set.

In the answer cell: what is the 50% step length? Why does the model make many errors for control subjects? Compare the test accuracy with the always-PD model.

In [ ]:
# 1. step length where p(PD) = 0.5  (z = 0)
x50 = -b0 / b1
print(f'step length at p(PD) = 0.5: {x50:.1f} cm')

# 2. probability curve + train recordings
x_curve = pd.DataFrame({'step_length_cm': np.linspace(30, 100, 300)})
p_curve = model_A1.predict_proba(x_curve)[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(x_curve['step_length_cm'], p_curve, lw=2, label='Model: probability of PD')
plt.scatter(train['step_length_cm'], y_train, alpha=0.3, color='k', label='Train recordings (0 = control, 1 = PD)')
plt.axvline(x50, color='red', ls='--', label=f'50% point ({x50:.1f} cm)')
plt.xlim(30, 100)
plt.xlabel('Step length (cm)')
plt.ylabel('Probability of PD')
plt.title('Logistic regression: probability of PD vs step length')
plt.legend()
plt.grid()
plt.show()

# 3. accuracy
print(f'train accuracy: {model_A1.score(train[["step_length_cm"]], y_train):.3f}')
print(f'test accuracy:  {model_A1.score(test[["step_length_cm"]], y_test):.3f}')
print(f'always-PD model: {always_pd_accuracy:.3f}')

pred_A1 = model_A1.predict(test[['step_length_cm']])
print(pd.crosstab(y_test, pred_A1, rownames=['true class'], colnames=['predicted class']))

**Your answer:** The 50% step length is **70.5 cm**. The model predicts PD for any recording with a step length below 70.5 cm and control above it.

The model makes many errors for controls because the two classes overlap a lot. Many control subjects also walk with steps shorter than 70.5 cm, and the controls in this dataset have a mean step length of about 69 cm, right at the boundary. A single cut-off on step length cannot separate them. Also, there are more than twice as many PD recordings as controls, so the model gains more by getting PD right. On the test set, it labels **12 of the 23 controls as PD** but misses only 7 of the 54 PD recordings.

The test accuracy is **0.753**, against **0.698** for the always-PD model (train accuracy 0.750). That is only about 5 percentage points better, so step length alone carries some information about PD, but not much.

### Part (c) - Odds ratio: what does the coefficient really mean?

Research papers report the coefficient of a logistic regression as an **odds ratio**: $\text{OR} = e^{\beta_1}$ per unit of the input (lecture 04.01).

1. Print the odds ratio of step length per 1 cm and per 10 cm ($e^{10\beta_1}$).
2. Make a table for step lengths of **50, 60, 70, 80 and 90 cm**. For each step length, compute with `model_A1.predict_proba`:
   - the probability of PD, $p$
   - the odds of PD, $p / (1 - p)$

   Then, for each 10 cm step (50 to 60, 60 to 70, ...), compute the **change in probability** and the **ratio of the odds** (odds after / odds before).

   **Hint:** put the step lengths in a DataFrame, add columns with the probability and the odds, then use `.diff()` for the change in probability and `.shift()` for the ratio of the odds:
   ```Python
   table['change in p'] = table['p'].diff()
   table['odds ratio']  = table['odds'] / table['odds'].shift()
   ```

In the answer cell:
- Does the ratio of the odds change from one 10 cm step to the next? Does the change in probability change? Explain the difference with the shape of the sigmoid.
- A news article about this model says: *"Each 10 cm of extra step length lowers the risk of Parkinson's disease by 62%."* What is wrong with this sentence? Write a correct version.
- Why is the odds ratio per 10 cm more useful to report than the odds ratio per 1 cm?

In [ ]:
# 1. odds ratios
print(f'odds ratio per 1 cm:  e^(b1)    = {np.exp(b1):.3f}')
print(f'odds ratio per 10 cm: e^(10*b1) = {np.exp(10 * b1):.3f}')

# 2. table of probability and odds for 50 to 90 cm
table = pd.DataFrame({'step_length_cm': [50, 60, 70, 80, 90]})
table['p'] = model_A1.predict_proba(table[['step_length_cm']])[:, 1]
table['odds'] = table['p'] / (1 - table['p'])
table['change in p'] = table['p'].diff()
table['odds ratio']  = table['odds'] / table['odds'].shift()
table.round(3)

**Your answer:** The odds ratio is **0.907 per 1 cm** and **0.377 per 10 cm**.

- **Ratio of the odds:** it does **not** change. Every 10 cm step multiplies the odds of PD by the same 0.377, whether you go from 50 to 60 cm or from 80 to 90 cm.
- **Change in probability:** it **does** change: −0.145, −0.224, −0.229, −0.154. It is largest around 70–80 cm, near the 50% point, and smaller at the ends.
- **Why:** logistic regression is linear in the log-odds ($z$), so a fixed change in $x$ always multiplies the odds by the same factor. The probability, however, goes through the S-shaped sigmoid. The sigmoid is steepest in the middle ($p$ near 0.5) and flat near 0 and 1, so the same change in odds moves $p$ a lot in the middle and only a little at the ends.

**What is wrong with the news sentence:**

1. **Odds are not risk.** The 62% (1 − 0.377) is a drop in the **odds**, not in the probability or "risk". As the table shows, the drop in probability is not 62% and depends on where you start.
2. **Association is not cause.** The model describes recordings that already exist. Taking longer steps will not "lower your risk" of getting Parkinson's disease; the short steps are a consequence of the disease.

A correct version: *"In this sample, each extra 10 cm of step length was associated with 62% lower odds that the recording came from a person with Parkinson's disease (odds ratio 0.38 per 10 cm)."*

**Why report per 10 cm:** a 1 cm change in step length is too small to matter in practice, and its odds ratio (0.907) is close to 1, which makes the effect hard to grasp. 10 cm is a realistic, meaningful difference between people (the SD of step length is about 13 cm), so the odds ratio per 10 cm describes an effect you would actually see.

## Question A3 (0.5 points) - Two inputs: step length and step time variability

People with PD often walk with less regular steps. Train a `LogisticRegression` with the inputs `['step_length_cm', 'cv_step_time_pct']`.

1. Print the two coefficients and the intercept, and the accuracy on the train set and on the test set.
2. Make a scatter plot of the **test** set (step length on the x-axis, variability on the y-axis, one color per class). Draw the decision boundary, where $\beta_0 + \beta_1 x_1 + \beta_2 x_2 = 0$:
   ```Python
   x1 = np.linspace(30, 100, 100)
   x2 = -(b0 + b1 * x1) / b2
   plt.plot(x1, x2)
   plt.xlim(30, 100)
   plt.ylim(0, 80)   # a few recordings have extreme values; this keeps the plot readable
   ```

In the answer cell: on which side of the line does the model predict PD? Do the signs of the coefficients agree with PD gait? Did the second input improve the test accuracy?

In [ ]:
inputs_A2 = ['step_length_cm', 'cv_step_time_pct']

model_A2 = LogisticRegression()
model_A2.fit(train[inputs_A2], y_train)

b0_2 = model_A2.intercept_[0]
b1_2, b2_2 = model_A2.coef_[0]
print(f'coefficient of step_length_cm:   {b1_2:.4f}')
print(f'coefficient of cv_step_time_pct: {b2_2:.4f}')
print(f'intercept: {b0_2:.4f}')
print(f'train accuracy: {model_A2.score(train[inputs_A2], y_train):.3f}')
print(f'test accuracy:  {model_A2.score(test[inputs_A2], y_test):.3f}')

# scatter of the test set, one color per class, and the decision boundary
plt.figure(figsize=(8, 6))
for cls, name, color in [(0, 'control', 'tab:blue'), (1, 'PD', 'tab:red')]:
    rows = test[y_test == cls]
    plt.scatter(rows['step_length_cm'], rows['cv_step_time_pct'], color=color, alpha=0.7, label=name)

x1 = np.linspace(30, 100, 100)
x2 = -(b0_2 + b1_2 * x1) / b2_2
plt.plot(x1, x2, color='k', lw=2, label='Decision boundary (p = 0.5)')
plt.xlim(30, 100)
plt.ylim(0, 80)   # a few recordings have extreme values; this keeps the plot readable
plt.xlabel('Step length (cm)')
plt.ylabel('Step time variability, CV (%)')
plt.title('Test set: control vs PD')
plt.legend()
plt.grid()
plt.show()

**Your answer:** The model is $z$ = 6.210 − 0.1017 × step length + 0.0511 × CV of step time.

- **Where PD is predicted:** above and to the left of the line, where steps are short and/or step timing is very variable. Control is predicted below and to the right: long, regular steps.
- **Signs of the coefficients:** they agree with PD gait. Step length has a negative coefficient (shorter steps mean more likely PD), and step time variability has a positive one (less regular steps mean more likely PD).
- **Effect of the second input:** test accuracy rose from 0.753 (step length only) to **0.805**, and train accuracy from 0.750 to 0.776. So the second input helped. The plot also shows that many recordings remain mixed near the line, especially controls with short steps.

## Question A4 (0.75 points) - All inputs

Now use all 16 gait measures in `features`. They have very different units, so scale them first, as in Assignment 6. Answer parts (a) to (c), each in its own cell.

### Part (a) - Fit, evaluate and count the errors

```Python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model_A3 = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=5000))])
```

Train it on `train[features]`. Print the train and test accuracy. Then count the errors for each class on the test set:

```Python
pred = model_A3.predict(test[features])
print(pd.crosstab(y_test, pred, rownames=['true class'], colnames=['predicted class']))
```

In the answer cell: compare the test accuracy with Question A3. How many PD recordings did the model miss? How many controls did it label as PD? Why can one accuracy number hide a problem?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model_A3 = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=5000))])
model_A3.fit(train[features], y_train)

print(f'train accuracy: {model_A3.score(train[features], y_train):.3f}')
print(f'test accuracy:  {model_A3.score(test[features], y_test):.3f}')

pred = model_A3.predict(test[features])
print(pd.crosstab(y_test, pred, rownames=['true class'], colnames=['predicted class']))

**Your answer:** The test accuracy is **0.779**, **lower** than the 0.805 of the two-input model in Question A3. The train accuracy is much higher (0.855), so the 16-input model fits the training data better but does not generalize better. That is a sign of overfitting.

From the errors table:

- The model **missed 4 PD recordings** (labelled control).
- It **labelled 13 of the 23 controls as PD**, so it gets fewer than half of the controls right.

One accuracy number hides this because the classes are unbalanced: 54 of the 77 test recordings are PD. The model can reach about 78% mostly by getting the PD recordings right, while being wrong for most controls. The accuracy mixes both types of error into one number. It does not say which class the errors fall on, and the two kinds of error can have very different consequences.

### Part (b) - Change the threshold

`predict` says PD when the probability of PD is 0.5 or more. You can choose a different threshold. Get the probabilities with `prob = model_A3.predict_proba(test[features])[:, 1]`. For each threshold in `[0.3, 0.5, 0.7]`, make the prediction with `pred = (prob >= threshold).astype(int)` and print the accuracy, the number of missed PD recordings and the number of false alarms (controls labeled as PD).

In the answer cell: what happens to the two types of error when the threshold increases? A clinic uses this model as a **screening tool**: a recording labeled PD goes to a neurologist for a full exam. Which threshold would you choose, and why?

In [ ]:
prob = model_A3.predict_proba(test[features])[:, 1]   # probability of PD

for threshold in [0.3, 0.5, 0.7]:
    pred = (prob >= threshold).astype(int)
    accuracy = (pred == y_test).mean()
    missed_pd = ((y_test == 1) & (pred == 0)).sum()
    false_alarms = ((y_test == 0) & (pred == 1)).sum()
    print(f'threshold {threshold}: accuracy = {accuracy:.3f}, missed PD = {missed_pd}, false alarms = {false_alarms}')

**Your answer:**

| Threshold | Accuracy | Missed PD | False alarms |
| --- | --- | --- | --- |
| 0.3 | 0.818 | 1 | 13 |
| 0.5 | 0.779 | 4 | 13 |
| 0.7 | 0.779 | 12 | 5 |

When the threshold **increases**, the model needs to be more sure before it says PD. So it says PD less often: **missed PD recordings go up** (1 → 4 → 12) and **false alarms go down** (13 → 13 → 5). The two errors trade against each other.

For a **screening tool**, I would choose the **lowest threshold, 0.3**. The cost of a miss is high: a person with PD goes home without a diagnosis or treatment. The cost of a false alarm is low: a healthy person gets a full exam from a neurologist, who then finds no PD. At 0.3, the model misses only 1 of 54 PD recordings. It does produce 13 false alarms, but the neurologist's exam catches those. Here 0.3 also happens to give the highest accuracy.

### Part (c) - PD or slow walking?

People with PD often walk slower than controls, and many gait measures change with speed. Print the mean `avg_speed` of each class. Train a `LogisticRegression` with `avg_speed` as the only input and print its test accuracy. Compare it with `model_A3`, which does **not** use speed.

In the answer cell: what does this comparison tell you about what `model_A3` learned?

In [ ]:
print('mean walking speed per class (m/s):')
print(data.groupby('class')['avg_speed'].mean())

model_speed = LogisticRegression()
model_speed.fit(train[['avg_speed']], y_train)
print(f'\nspeed-only model: test accuracy = {model_speed.score(test[["avg_speed"]], y_test):.3f}')
print(f'model_A3 (16 gait measures, no speed): test accuracy = {model_A3.score(test[features], y_test):.3f}')

**Your answer:** Controls walk at **1.24 m/s** on average and PD subjects at **1.02 m/s**. A model with **speed as its only input** reaches a test accuracy of **0.766**, almost the same as the 0.779 of `model_A3` with all 16 gait measures.

This suggests that `model_A3` mostly learned **how fast a person walks**, not something specific to Parkinson's disease. Speed is not one of its inputs, but it is hidden in them: step length and cadence together give speed (speed = cadence × step length / 60), and the timing measures also change with speed. The 15 extra measures add only about one percentage point beyond what speed alone gives. So the model may be partly a "slow walker" detector. A healthy person who walks slowly would likely be flagged as PD, and a person with PD who walks fast would be missed. To learn PD-specific features, we would need to compare subjects at the same speed or correct the measures for speed.

---
# Part B - Running speed (2.5 points)

## The data

You used this dataset in Assignment 5: 28 volunteers ran on a treadmill at **2.5, 3.5 and 4.5 m/s** (Fukuchi et al., 2017, *PeerJ* 5, e3298). The file `gaitData.zip` is in the `Assignments` folder. Each participant has one file with one gait cycle of each joint at each speed.

**Goal:** predict the running speed of a stride from the motion of the joints. The output is one of **three classes**: `2.5`, `3.5` or `4.5`. In lecture 04.01 you saw that logistic regression works with more than two classes: it uses the **softmax** function instead of the sigmoid, and it gives one probability for each class.

Run the cell below. It reads all 28 files and makes a table with one row per participant and speed (28 x 3 = 84 rows). For each row it computes four values from the **right** leg that you know from Assignment 5:

| Column | Contents |
| --- | --- |
| `subject` | participant number, 1 to 28 |
| `speed` | running speed, as text: `'2.5'`, `'3.5'` or `'4.5'` |
| `peak_hip_flex` | maximum hip flexion (sagittal angle), in degrees |
| `peak_knee_flex` | maximum knee flexion (sagittal angle), in degrees |
| `peak_ankle_pow` | maximum ankle power, in W/kg |
| `peak_knee_pow` | maximum knee power, in W/kg |

In [ ]:
import zipfile
zipfile.ZipFile('gaitData.zip').extractall('.')      # creates the gaitData/ folder

rows = []
for subject in range(1, 29):
    gait = pd.read_csv(f'gaitData/RBDS{subject:03d}processed.txt', sep='\t')
    for s in ['25', '35', '45']:
        rows.append({
            'subject':        subject,
            'speed':          s[0] + '.' + s[1],                 # '25' -> '2.5'
            'peak_hip_flex':  gait[f'RhipAngZ{s}'].max(),
            'peak_knee_flex': gait[f'RkneeAngZ{s}'].max(),
            'peak_ankle_pow': gait[f'RanklePow{s}'].max(),
            'peak_knee_pow':  gait[f'RkneePow{s}'].max(),
        })

running = pd.DataFrame(rows)
running.head()

## Question B1 (0.25 points) - Prepare the data

1. Print the number of rows for each speed. What is the accuracy of a model that always predicts the same speed?
2. Split the data **by subject**: subjects 1 to 21 go to the train set, subjects 22 to 28 to the test set.
   ```Python
   train_B = running[running['subject'] <= 21]
   test_B  = running[running['subject'] > 21]
   y_train_B, y_test_B = train_B['speed'], test_B['speed']
   ```
   Print the number of rows in each set.

In the answer cell: in Part A you split the rows at random. Here each subject has three rows (one per speed). Why is it better to keep all the rows of one subject in the same set?

In [ ]:
# 1. rows per speed and accuracy of an always-same-speed model
print(running['speed'].value_counts())
always_same_accuracy = running['speed'].value_counts().max() / len(running)
print(f'accuracy of a model that always predicts the same speed: {always_same_accuracy:.3f}')

# 2. split by subject
train_B = running[running['subject'] <= 21]
test_B  = running[running['subject'] > 21]
y_train_B, y_test_B = train_B['speed'], test_B['speed']
print(f'\ntrain rows: {len(train_B)} (subjects 1-21), test rows: {len(test_B)} (subjects 22-28)')

**Your answer:** There are 28 rows for each speed (84 in total), so a model that always predicts the same speed is right 28 / 84 = **0.333 (one third)** of the time. The train set has 63 rows (21 subjects × 3 speeds) and the test set 21 rows (7 subjects × 3 speeds).

Keeping all the rows of a subject in the same set prevents **data leakage**. The three rows of one runner are strongly linked: the same body, the same running style, the same typical joint angles. If some rows of a runner were in the train set and others in the test set, the model would already "know" that runner. It could score well partly by recognizing the person, not by learning how speed changes the motion. The test accuracy would then be too optimistic. Splitting by subject means the test runners are truly new people, which is how the model would be used in practice.

## Question B2 (0.75 points) - One input: hip flexion

In Assignment 5 you saw that the peak hip flexion increases with speed. Answer parts (a) and (b), each in its own cell.

### Part (a) - Fit the model and compute the probabilities by hand

Train a `LogisticRegression` on the train set with `peak_hip_flex` as the only input. Print `model.classes_`, `model.coef_` and `model.intercept_`. With three classes, the model has **one coefficient and one intercept for each class**.

For a stride with a peak hip flexion of **53 degrees**:

1. Compute the probabilities **by hand** with the softmax function from lecture 04.01. For each class $k$: $z_k = \beta_{0,k} + \beta_{1,k} x$. Then $p_k = e^{z_k} / (e^{z_1} + e^{z_2} + e^{z_3})$.
2. Check your result with `model.predict_proba(pd.DataFrame({'peak_hip_flex': [53]}))`.

In the answer cell: compare this with Question A2. How many coefficients did you get there, and how many here? What is the sum of the three probabilities? Which speed does the model predict for 53 degrees?

In [ ]:
model_B1 = LogisticRegression()
model_B1.fit(train_B[['peak_hip_flex']], y_train_B)

print('classes:   ', model_B1.classes_)
print('coef_:     ', model_B1.coef_.ravel())
print('intercept_:', model_B1.intercept_)

# softmax by hand for a peak hip flexion of 53 degrees
x = 53
z = model_B1.intercept_ + model_B1.coef_.ravel() * x     # one z per class
p = np.exp(z) / np.exp(z).sum()
for cls, zk, pk in zip(model_B1.classes_, z, p):
    print(f'{cls} m/s: z = {zk:.3f}, p = {pk:.4f}')
print(f'sum of the three probabilities: {p.sum():.4f}')

print('\npredict_proba:', model_B1.predict_proba(pd.DataFrame({'peak_hip_flex': [53]})))
print('predicted speed:', model_B1.predict(pd.DataFrame({'peak_hip_flex': [53]}))[0], 'm/s')

**Your answer:** In Question A2 (two classes), the model had **one coefficient and one intercept**: one $z$, turned into $p$(PD) by the sigmoid, with $p$(control) = 1 − $p$(PD). Here, with three classes, there are **three coefficients and three intercepts**, one $z_k$ per speed:

- $\beta_1$ = −0.215 for 2.5 m/s, 0.009 for 3.5 m/s and 0.206 for 4.5 m/s.
- $\beta_0$ = 10.93, 0.17 and −11.10.

The softmax turns the three $z_k$ into three probabilities. For 53°, I get 0.186 (2.5 m/s), 0.559 (3.5 m/s) and 0.255 (4.5 m/s), and `predict_proba` gives the same values. The three probabilities **sum to 1**. The model predicts **3.5 m/s**, the class with the highest probability. That makes sense, because 53° is close to the mean peak hip flexion of the 3.5 m/s strides (about 53°).

### Part (b) - The probability curves

Plot the probability of each speed for peak hip flexion values from 30 to 75 degrees: three curves on the same axes, with a legend. Print the accuracy on the train set and on the test set.

In the answer cell: describe the three curves. Which speed is most probable for a small, a medium and a large hip flexion? Compare the test accuracy with the always-same-speed model.

In [ ]:
hip_range = pd.DataFrame({'peak_hip_flex': np.linspace(30, 75, 300)})
probs = model_B1.predict_proba(hip_range)

plt.figure(figsize=(8, 5))
for k, speed in enumerate(model_B1.classes_):
    plt.plot(hip_range['peak_hip_flex'], probs[:, k], lw=2, color=f'C{k}', label=f'{speed} m/s')
plt.xlabel('Peak hip flexion (deg)')
plt.ylabel('Probability')
plt.title('Probability of each running speed vs peak hip flexion')
plt.legend()
plt.grid()
plt.show()

print(f'train accuracy: {model_B1.score(train_B[["peak_hip_flex"]], y_train_B):.3f}')
print(f'test accuracy:  {model_B1.score(test_B[["peak_hip_flex"]], y_test_B):.3f}')
print(f'always-same-speed model: {always_same_accuracy:.3f}')

**Your answer:**

- **2.5 m/s curve:** starts near 1 at small hip flexion and falls as hip flexion grows.
- **4.5 m/s curve:** the mirror image. Near 0 at small hip flexion, it rises to near 1 at large hip flexion.
- **3.5 m/s curve:** a hump in the middle. It is highest (about 0.56) around 52–53° and lower at both ends.

At every hip flexion value, the three curves add up to 1. The most probable speed is **2.5 m/s for small hip flexion** (below about 48°), **3.5 m/s for medium hip flexion** (about 48–57°), and **4.5 m/s for large hip flexion** (above about 57°). The middle class never gets a very high probability, because its hip flexion values overlap with both neighbours.

The test accuracy is **0.714** (train 0.651), more than double the **0.333** of the always-same-speed model. So one joint angle already carries a lot of information about speed.

## Question B3 (0.5 points) - Two inputs: hip flexion and ankle power

Train a `LogisticRegression` with the inputs `['peak_hip_flex', 'peak_ankle_pow']`. Print the accuracy on the train set and on the test set.

With three classes, the boundaries between the classes are no longer one line. Run this code to color each region of the plot with the speed that the model predicts, and to draw the **test** strides on top:

```Python
inputs_B2 = ['peak_hip_flex', 'peak_ankle_pow']
h, a = np.meshgrid(np.linspace(30, 75, 200), np.linspace(3, 17, 200))
grid = pd.DataFrame({'peak_hip_flex': h.ravel(), 'peak_ankle_pow': a.ravel()})
region = model_B2.predict(grid).astype(float).reshape(h.shape)

plt.contourf(h, a, region, levels=[2, 3, 4, 5], colors=['C0', 'C1', 'C2'], alpha=0.2)
for k, speed in enumerate(['2.5', '3.5', '4.5']):
    rows = test_B[y_test_B == speed]
    plt.scatter(rows['peak_hip_flex'], rows['peak_ankle_pow'], color=f'C{k}', label=f'{speed} m/s')
plt.xlabel('Peak hip flexion (deg)')
plt.ylabel('Peak ankle power (W/kg)')
plt.legend()
plt.show()
```

In the answer cell: did the second input improve the test accuracy? Look at the plot: where are the errors? Compare the shape of the regions with the single line of Question A3.

In [ ]:
inputs_B2 = ['peak_hip_flex', 'peak_ankle_pow']

model_B2 = LogisticRegression(max_iter=1000)
model_B2.fit(train_B[inputs_B2], y_train_B)
print(f'train accuracy: {model_B2.score(train_B[inputs_B2], y_train_B):.3f}')
print(f'test accuracy:  {model_B2.score(test_B[inputs_B2], y_test_B):.3f}')

h, a = np.meshgrid(np.linspace(30, 75, 200), np.linspace(3, 17, 200))
grid = pd.DataFrame({'peak_hip_flex': h.ravel(), 'peak_ankle_pow': a.ravel()})
region = model_B2.predict(grid).astype(float).reshape(h.shape)

plt.figure(figsize=(8, 6))
plt.contourf(h, a, region, levels=[2, 3, 4, 5], colors=['C0', 'C1', 'C2'], alpha=0.2)
for k, speed in enumerate(['2.5', '3.5', '4.5']):
    rows = test_B[y_test_B == speed]
    plt.scatter(rows['peak_hip_flex'], rows['peak_ankle_pow'], color=f'C{k}', label=f'{speed} m/s')
plt.xlabel('Peak hip flexion (deg)')
plt.ylabel('Peak ankle power (W/kg)')
plt.title('Predicted speed regions and test strides')
plt.legend()
plt.show()

print(pd.crosstab(y_test_B, model_B2.predict(test_B[inputs_B2]), rownames=['true speed'], colnames=['predicted speed']))

**Your answer:** Yes. The test accuracy rose from 0.714 (hip flexion only) to **0.810** (17 of 21 strides), and the train accuracy from 0.651 to 0.905. Ankle power separates the speeds well, because it rises strongly with speed (Assignment 5).

The **4 errors** are all between **neighbouring speeds**, at the borders between regions:

- one 2.5 m/s stride predicted as 3.5 m/s
- one 3.5 m/s stride predicted as 2.5 m/s
- two 4.5 m/s strides predicted as 3.5 m/s: these runners (subjects 22 and 28) have both a low hip flexion (52–55° vs a mean of about 61°) and a low ankle power (about 9.2–9.5 vs 11.1 W/kg) for that speed, so they fall in the 3.5 m/s region.

Subject 22 accounts for two of the four errors: this runner moves with smaller hip flexion and ankle power than most people at every speed, so the model shifts them down one speed class.

**Shape of the regions:** in Question A3, a single straight line split the plane into two halves (PD or control). Here there are **three regions**, separated by **two straight boundary lines**. Each line is where two classes have the same probability. The regions run roughly diagonally in bands from low hip flexion and low ankle power (2.5 m/s) to high values (4.5 m/s). The boundaries are still straight, because the model is linear in its inputs, but with three classes there are several of them.

## Question B4 (1 point) - More inputs, and what the two parts teach

Answer parts (a) to (c), each in its own cell.

### Part (a) - All four inputs

Use all four measures: `['peak_hip_flex', 'peak_knee_flex', 'peak_ankle_pow', 'peak_knee_pow']`. They have different units, so use the same pipeline as in Question A4 (`StandardScaler` + `LogisticRegression`). Print the train and test accuracy, and the errors table:

```Python
pred_B = model_B3.predict(test_B[inputs_B3])
print(pd.crosstab(y_test_B, pred_B, rownames=['true speed'], colnames=['predicted speed']))
```

In the answer cell: compare the test accuracy with Question B3. Does the model ever confuse 2.5 m/s with 4.5 m/s? Why does this make sense?

In [ ]:
inputs_B3 = ['peak_hip_flex', 'peak_knee_flex', 'peak_ankle_pow', 'peak_knee_pow']

model_B3 = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=5000))])
model_B3.fit(train_B[inputs_B3], y_train_B)
print(f'train accuracy: {model_B3.score(train_B[inputs_B3], y_train_B):.3f}')
print(f'test accuracy:  {model_B3.score(test_B[inputs_B3], y_test_B):.3f}')

pred_B = model_B3.predict(test_B[inputs_B3])
print(pd.crosstab(y_test_B, pred_B, rownames=['true speed'], colnames=['predicted speed']))

**Your answer:** The test accuracy is **0.810**, exactly the same as the two-input model in Question B3. It even makes the same 4 errors. The train accuracy is 0.889. So adding knee flexion and knee power did not improve the test result.

The model **never confuses 2.5 m/s with 4.5 m/s**. All errors are between neighbouring speeds (2.5 ↔ 3.5 and 3.5 ↔ 4.5). This makes sense because the joint measures change gradually with speed: the 2.5 and 4.5 m/s strides are far apart (for example, mean peak hip flexion is about 44° vs 61°, and peak ankle power about 6 vs 11 W/kg), while each one overlaps with 3.5 m/s. To be mistaken for 4.5 m/s, a 2.5 m/s stride would need very extreme values, and no runner has them.

### Part (b) - How much can you trust one test accuracy?

Your test set has only 7 runners (21 rows). A different choice of 7 runners can give a different accuracy. Run this experiment for the two-input model (`inputs_B2`) and the four-input model (`inputs_B3`). It repeats the split by subject 20 times, each time with 7 different test runners chosen at random:

```Python
rng = np.random.RandomState(0)
accuracies = []
for repeat in range(20):
    test_subjects = rng.choice(np.arange(1, 29), 7, replace=False)
    tr = running[~running['subject'].isin(test_subjects)]
    te = running[running['subject'].isin(test_subjects)]
    model = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=5000))])
    model.fit(tr[inputs], tr['speed'])
    accuracies.append(model.score(te[inputs], te['speed']))
```

Put this code in a `for` loop over the two input lists. For each one, print the mean, the minimum and the maximum of the 20 test accuracies.

In the answer cell: how much does the test accuracy change from one split to the next? On average, which model is better? Can you trust the difference between the models that you found in part (a) and Question B3, with one split only?

In [ ]:
for name, inputs in [('two inputs (inputs_B2)', inputs_B2), ('four inputs (inputs_B3)', inputs_B3)]:
    rng = np.random.RandomState(0)
    accuracies = []
    for repeat in range(20):
        test_subjects = rng.choice(np.arange(1, 29), 7, replace=False)
        tr = running[~running['subject'].isin(test_subjects)]
        te = running[running['subject'].isin(test_subjects)]
        model = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=5000))])
        model.fit(tr[inputs], tr['speed'])
        accuracies.append(model.score(te[inputs], te['speed']))
    accuracies = np.array(accuracies)
    print(f'{name}: mean = {accuracies.mean():.3f}, min = {accuracies.min():.3f}, max = {accuracies.max():.3f}')

**Your answer:** The test accuracy changes a lot from one split to the next. For both models it ranges from **0.667 to 1.000**, depending only on which 7 runners land in the test set. With 21 test strides, one stride more or less is about 5 percentage points.

On average, the **two-input model is better**: a mean of **0.836** against **0.805** for the four-input model.

No, I **cannot trust** the difference (or the absence of a difference) from a single split. In part (a) and Question B3, both models scored 0.810 on the one split. That number could just as easily have been 0.67 or 1.00, and on other splits either model could have come out ahead. The split-to-split spread (about 0.33) is ten times larger than the difference between the models (about 0.03). Only the average over many splits, as in cross-validation, gives a reliable comparison. Even then, the evidence is that the two extra knee measures do not help.

### Part (c) - Same technique, two datasets

In the answer cell, compare Part A and Part B:

- What was the **same** in your code for the two parts?
- What **changed** when you went from two classes to three classes (number of coefficients, the function that gives the probabilities, the number of columns of `predict_proba`, the decision boundaries)?
- In each part, did more inputs always give a better test accuracy?

**Your answer:**

**What stayed the same**
- The same `LogisticRegression` model and the same scikit-learn steps: `fit`, `predict`, `predict_proba`, `score`.
- The same workflow: train/test split, a trivial baseline to beat, one input, then two, then more, with `StandardScaler` in a `Pipeline` when the inputs have different units.
- The same evaluation tools: accuracy, the errors table (`pd.crosstab`) and decision-boundary plots.
- In both parts, the model is linear in its inputs: it computes $z = \beta_0 + \beta_1 x_1 + \dots$ and turns it into probabilities.

**What changed from two to three classes**

| | Two classes (Part A) | Three classes (Part B) |
| --- | --- | --- |
| Coefficients | one set: one $\beta_0$ and one $\beta$ per input | one set **per class**: 3 intercepts and 3 $\beta$ per input |
| Function giving the probabilities | sigmoid, $p = 1/(1+e^{-z})$ | softmax, $p_k = e^{z_k} / \sum_j e^{z_j}$ |
| Columns of `predict_proba` | 2 (control, PD) | 3 (2.5, 3.5, 4.5 m/s) |
| Decision boundaries | one straight line, two regions | several straight lines, three regions |
| Prediction | PD if $p \geq 0.5$ (threshold can be changed) | class with the highest probability |

**Did more inputs always help?** **No**, in neither part.
- **Part A:** one input gave 0.753, two inputs gave 0.805, and all 16 inputs gave only 0.779 (lower than two inputs, with a much higher train accuracy, so overfitting).
- **Part B:** one input gave 0.714 and two inputs gave 0.810, but four inputs gave the same 0.810. Over 20 random splits, the four-input model was on average *worse* (0.805 vs 0.836).

In both parts, the second, well-chosen input gave the real improvement. Adding more inputs mainly added complexity, and with a small dataset that leads to overfitting.